In [16]:
from geopy.distance import geodesic
import json
from itertools import combinations
import pickle
import bm25s
from tqdm import tqdm

In [2]:
def calculate_distance(lat1, long1, lat2, long2):
    coord1 = (lat1, long1)
    coord2 = (lat2, long2)
    return geodesic(coord1, coord2).kilometers

def load_json(path):
    with open(path, 'r') as f:
        data = json.load(f)
    return data

## Distance between CITIES

In [3]:
city_data = load_json('/projects/iris/rferreir/datasets/TourismQA/original_data/final_cities_lat_long.json')

In [14]:
cities_comb = list(combinations(city_data, 2))

In [15]:
d_dic = {}
for c1 in city_data:
    c_dic = {}
    for c2 in city_data:
        if c1["city_id"] != c2["city_id"]:
            c_dic[c2['city_name']] = 0
    d_dic[c1['city_name']] = c_dic
    

In [16]:
for c1, c2 in cities_comb:
    lat1 = c1['location']['lat']
    lat2 = c2['location']['lat']
    long1 = c1['location']['lng']
    long2 = c2['location']['lng']
    d = calculate_distance(lat1, long1, lat2, long2)
    d_dic[c1['city_name']][c2['city_name']] = d
    d_dic[c2['city_name']][c1['city_name']] = d

In [18]:
with open("/projects/iris/rferreir/datasets/TourismQA/original_data/d_city_to_city.pkl", 'wb') as f:
    pickle.dump(d_dic, f)

## Map POI to CITY

In [5]:
poi_data = load_json('/projects/iris/rferreir/datasets/TourismQA/original_data/TourQue_Knowledge_SelSum.json')
print(len(poi_data))

114520


In [ ]:
city_pois = {}
for c in city_data:
    city_pois[c['city_name']] = []
for idx, poi in tqdm(poi_data.items()):
    if 'name' in poi:
        city = None
        p_name = poi['name'].lower()
        for c in city_data:
            c_name = c['city_name']
            if c_name in p_name:
                city = c['city_name']
        if city:
            city_pois[city].append(idx)
        else:
            print(poi['name'])
        

In [6]:
with open("/projects/iris/rferreir/datasets/TourismQA/original_data/city_pois.pkl", 'wb') as f:
    pickle.dump(city_pois, f)

## Creating a BM25 index

In [17]:
with open("/projects/iris/rferreir/datasets/TourismQA/original_data/city_pois.pkl", 'rb') as f:
    city_pois = pickle.load(f)
for c in city_pois:
    tmp = {}
    tmp['R'] = set()
    tmp['H'] = set()
    tmp['A'] = set()
    for idx in city_pois[c]:
        for letter in ['R', 'H', 'A']:
            if letter in idx:
                tmp[letter].add(idx)
                break
    city_pois[c] = tmp

In [19]:
import time
nb = 0
for c in city_pois:
    for letter in ['R', 'A', 'H']:
        corpus = []
        poi_list = list(city_pois[c][letter])
        if len(poi_list) == 0:
            continue
        for poi in poi_list:
            corpus.append(" ".join(poi_data[poi]['review']))
        corpus_tokens = bm25s.tokenize(corpus, stopwords="en")

        # Create the BM25 model and index the corpus
        retriever = bm25s.BM25()
        retriever.index(corpus_tokens)

        for i, doc in enumerate(poi_list):
            corpus[i] = poi + ";;" + corpus[i]

        retriever.save(f"/projects/iris/rferreir/datasets/TourismQA/original_data/indexes/{c}_{letter}_index_bm25", corpus=corpus)
        nb += 1
        time.sleep(1)
print(nb)

Finding newlines for mmindex: 100%|██████████| 2.56M/2.56M [00:00<00:00, 82.4MB/s]
Finding newlines for mmindex: 100%|██████████| 221k/221k [00:00<00:00, 64.1MB/s]
Finding newlines for mmindex: 100%|██████████| 186k/186k [00:00<00:00, 92.4MB/s]
Finding newlines for mmindex: 100%|██████████| 718k/718k [00:00<00:00, 74.8MB/s]
Finding newlines for mmindex: 100%|██████████| 107k/107k [00:00<00:00, 64.5MB/s]
Finding newlines for mmindex: 100%|██████████| 83.7k/83.7k [00:00<00:00, 79.0MB/s]
Finding newlines for mmindex: 100%|██████████| 1.33M/1.33M [00:00<00:00, 74.5MB/s]
Finding newlines for mmindex: 100%|██████████| 126k/126k [00:00<00:00, 63.4MB/s]
Finding newlines for mmindex: 100%|██████████| 69.4k/69.4k [00:00<00:00, 72.5MB/s]
Finding newlines for mmindex: 100%|██████████| 1.06M/1.06M [00:00<00:00, 75.2MB/s]
Finding newlines for mmindex: 100%|██████████| 121k/121k [00:00<00:00, 58.2MB/s]
Finding newlines for mmindex: 100%|██████████| 119k/119k [00:00<00:00, 81.7MB/s]
Finding newlines f

147
